# 📊 Notebook 01 — Analyse Exploratoire des Données (EDA)
**Projet : Prédiction du Churn Client | 4IASD**

---
**Objectifs :**
- Comprendre la structure et la qualité du dataset
- Analyser la distribution des variables
- Identifier les patterns liés au churn
- Produire des visualisations exploitables pour le rapport

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print(' Imports OK')

## 1. Chargement du Dataset

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
print(f'Shape : {df.shape}')
df.head()

## 2. Aperçu Général

In [ ]:
print('=== Types de colonnes ===')
print(df.dtypes)
print()
print('=== Statistiques descriptives ===')
df.describe(include='all').T

## 3. Valeurs Manquantes

In [ ]:
# TotalCharges est souvent en string avec des espaces vides
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Manquants': missing, '% du total': missing_pct})
missing_df = missing_df[missing_df['Manquants'] > 0]

print('=== Valeurs manquantes ===')
print(missing_df if not missing_df.empty else 'Aucune valeur manquante détectée (hors TotalCharges converti)')

In [ ]:
# Visualisation des valeurs manquantes
fig, ax = plt.subplots(figsize=(8, 4))
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]

if not missing_counts.empty:
    missing_counts.plot(kind='bar', ax=ax, color='salmon')
    ax.set_title('Valeurs Manquantes par Colonne')
    ax.set_ylabel('Nombre de valeurs manquantes')
    plt.xticks(rotation=45)
else:
    ax.text(0.5, 0.5, 'Aucune valeur manquante\n(après conversion TotalCharges)',
            ha='center', va='center', fontsize=13, color='green')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/missing_values.png')
plt.show()

## 4. Distribution de la Variable Cible — Churn

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('=== Distribution du Churn ===')
for label in churn_counts.index:
    print(f'  {label} : {churn_counts[label]} clients ({churn_pct[label]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Countplot
sns.countplot(x='Churn', data=df, palette=['#4CAF50', '#F44336'], ax=axes[0])
axes[0].set_title('Distribution du Churn (Effectifs)')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Nombre de clients')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[1].set_title('Répartition du Churn (%)')

plt.suptitle('Déséquilibre des Classes — Variable Cible', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/churn_distribution.png')
plt.show()

## 5. Variables Démographiques vs Churn

In [ ]:
demo_vars = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(demo_vars):
    ct = df.groupby(col)['Churn'].value_counts(normalize=True).unstack() * 100
    ct.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#F44336'], edgecolor='white', width=0.6)
    axes[i].set_title(f'Churn par {col}')
    axes[i].set_ylabel('% de clients')
    axes[i].set_xlabel(col)
    axes[i].legend(['No Churn', 'Churn'], loc='upper right', fontsize=8)
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Variables Démographiques vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/demo_vs_churn.png')
plt.show()

## 6. Variables de Services vs Churn

In [ ]:
service_vars = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(service_vars):
    ct = df.groupby(col)['Churn'].value_counts(normalize=True).unstack() * 100
    ct.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#F44336'], edgecolor='white', width=0.6)
    axes[i].set_title(f'{col}', fontsize=10)
    axes[i].set_ylabel('% de clients', fontsize=8)
    axes[i].set_xlabel('')
    axes[i].legend(['No Churn', 'Churn'], fontsize=7)
    axes[i].tick_params(axis='x', rotation=20, labelsize=8)
    axes[i].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Variables de Services vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/services_vs_churn.png')
plt.show()

## 7. Variables d'Abonnement & Financières vs Churn

In [ ]:
contract_vars = ['Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(contract_vars):
    ct = df.groupby(col)['Churn'].value_counts(normalize=True).unstack() * 100
    ct.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#F44336'], edgecolor='white', width=0.6)
    axes[i].set_title(f'Churn par {col}')
    axes[i].set_ylabel('% de clients')
    axes[i].legend(['No Churn', 'Churn'], fontsize=8)
    axes[i].tick_params(axis='x', rotation=20, labelsize=8)
    axes[i].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Abonnement & Paiement vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/contract_vs_churn.png')
plt.show()

## 8. Variables Numériques — Distributions & Boxplots

In [ ]:
num_vars = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, col in enumerate(num_vars):
    # Distribution
    ax_hist = axes[0, i]
    for churn_val, color in zip(['No', 'Yes'], ['#4CAF50', '#F44336']):
        subset = df[df['Churn'] == churn_val][col].dropna()
        ax_hist.hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={churn_val}', edgecolor='white')
    ax_hist.set_title(f'Distribution — {col}')
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel('Fréquence')
    ax_hist.legend(fontsize=8)

    # Boxplot
    ax_box = axes[1, i]
    df.boxplot(column=col, by='Churn', ax=ax_box,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    ax_box.set_title(f'Boxplot — {col} par Churn')
    ax_box.set_xlabel('Churn')
    ax_box.set_ylabel(col)
    plt.sca(ax_box)
    plt.title(f'Boxplot — {col} par Churn')

plt.suptitle('Variables Numériques vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/numerical_vs_churn.png')
plt.show()

## 9. Heatmap de Corrélation (Variables Numériques)

In [ ]:
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_num']
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Heatmap de Corrélation — Variables Numériques', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png')
plt.show()

## 10. Analyse de la Tenure (Durée d'Abonnement)

In [ ]:
# Segmentation en groupes de tenure
df['tenure_group'] = pd.cut(df['tenure'],
                             bins=[0, 12, 24, 48, 72],
                             labels=['0-12 mois', '13-24 mois', '25-48 mois', '49-72 mois'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate par groupe de tenure
churn_by_tenure = df.groupby('tenure_group')['Churn'].apply(
    lambda x: (x == 'Yes').sum() / len(x) * 100
)
churn_by_tenure.plot(kind='bar', ax=axes[0], color='#F44336', edgecolor='white')
axes[0].set_title('Taux de Churn par Groupe de Tenure')
axes[0].set_ylabel('Taux de Churn (%)')
axes[0].set_xlabel('Groupe de Tenure')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Distribution tenure par churn
for churn_val, color in zip(['No', 'Yes'], ['#4CAF50', '#F44336']):
    axes[1].hist(df[df['Churn'] == churn_val]['tenure'],
                 bins=25, alpha=0.6, color=color, label=f'Churn={churn_val}', edgecolor='white')
axes[1].set_title('Distribution de la Tenure par Churn')
axes[1].set_xlabel('Tenure (mois)')
axes[1].set_ylabel('Nombre de clients')
axes[1].legend()

plt.suptitle('Analyse de la Durée d\'Abonnement (Tenure)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/tenure_analysis.png')
plt.show()

# Nettoyage colonne temporaire
df.drop(columns=['tenure_group'], inplace=True)

## 11. Analyse des Charges Financières

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MonthlyCharges par contrat et churn
df_temp = df[df['Churn'].isin(['Yes', 'No'])]
sns.violinplot(x='Contract', y='MonthlyCharges', hue='Churn',
               data=df_temp, split=True,
               palette={'No': '#4CAF50', 'Yes': '#F44336'},
               ax=axes[0])
axes[0].set_title('MonthlyCharges par Contrat & Churn')
axes[0].set_xlabel('Type de Contrat')
axes[0].set_ylabel('Frais Mensuels ($)')
axes[0].tick_params(axis='x', rotation=15)

# Scatter MonthlyCharges vs TotalCharges
colors = df['Churn'].map({'Yes': '#F44336', 'No': '#4CAF50'})
axes[1].scatter(df['tenure'], df['MonthlyCharges'],
                c=colors, alpha=0.3, s=10)
axes[1].set_title('Tenure vs MonthlyCharges (coloré par Churn)')
axes[1].set_xlabel('Tenure (mois)')
axes[1].set_ylabel('Frais Mensuels ($)')
# Légende manuelle
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#F44336', label='Churn=Yes'),
                   Patch(facecolor='#4CAF50', label='Churn=No')]
axes[1].legend(handles=legend_elements)

plt.suptitle('Analyse des Charges Financières', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/financial_analysis.png')
plt.show()

## 12. Synthèse — Insights Clés

In [ ]:
df['Churn_num'] = (df['Churn'] == 'Yes').astype(int)

churn_rate_global = df['Churn_num'].mean() * 100
churn_monthly = df[df['Contract'] == 'Month-to-month']['Churn_num'].mean() * 100
churn_fiber = df[df['InternetService'] == 'Fiber optic']['Churn_num'].mean() * 100
churn_no_security = df[df['OnlineSecurity'] == 'No']['Churn_num'].mean() * 100
churn_senior = df[df['SeniorCitizen'] == 1]['Churn_num'].mean() * 100
avg_tenure_churn = df[df['Churn'] == 'Yes']['tenure'].mean()
avg_monthly_churn = df[df['Churn'] == 'Yes']['MonthlyCharges'].mean()

print('='*55)
print('        SYNTHÈSE EDA — INSIGHTS CLÉS')
print('='*55)
print(f'  Taux de churn global          : {churn_rate_global:.1f}%')
print(f'  Churn contrat mensuel         : {churn_monthly:.1f}%')
print(f'  Churn Fiber optic             : {churn_fiber:.1f}%')
print(f'  Churn sans OnlineSecurity     : {churn_no_security:.1f}%')
print(f'  Churn clients seniors         : {churn_senior:.1f}%')
print(f'  Tenure moyen (churners)       : {avg_tenure_churn:.1f} mois')
print(f'  Frais mensuels moy (churners) : ${avg_monthly_churn:.2f}')
print('='*55)
print()
print(' Facteurs de risque principaux :')
print('   1. Contrat Month-to-month (forte flexibilité = fort churn)')
print('   2. Internet Fiber optic (clients plus exigeants)')
print('   3. Absence de services de sécurité (OnlineSecurity, TechSupport)')
print('   4. Tenure faible (< 12 mois = période critique)')
print('   5. Frais mensuels élevés sans engagement long terme')

df.drop(columns=['Churn_num'], inplace=True)

---
##  Conclusion EDA

L'analyse exploratoire met en évidence plusieurs profils à risque :

| Facteur | Impact sur le Churn |
|---------|--------------------|
| Contrat Month-to-month | Très élevé |
| Fiber optic sans services additionnels | Élevé |
| Tenure < 12 mois | Élevé |
| Frais mensuels > 70$ | Modéré |
| Client Senior sans partenaire | Modéré |

Ces insights guideront le **feature engineering** dans le notebook suivant (`02_preprocessing.ipynb`).